# 13 | 用 checkpoint 查看、回退和修正 LangGraph 状态

checkpoint 保存下来以后，不只是给下一轮自动恢复用。它还可以被查看、回到某个历史点继续运行，或者由人工直接修正 State。

checkpoint 的四个操作：

- `get_state`：查看当前 checkpoint；
- `get_state_history`：查看历史 checkpoint；
- 带 `checkpoint_id` 的 config：回到旧 checkpoint 继续运行；
- `update_state`：人工修改 checkpoint。

## 一、准备一个最小图

这个图做一件事：调用本地 LLM，根据 `command` 修改一条便签。

先导入相关依赖


In [ ]:
from importlib.metadata import version
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph

准备一个可以被 checkpoint 管理的最小 LangGraph。

它保存一条便签 `note`，每次根据新的 `command` 调用本地 LLM 改写便签。

`compile(checkpointer=...)` 会让运行后的 State 被保存下来，后面才能查看当前状态、查看历史、回到旧版本，以及手动修改状态。


In [ ]:
class NoteState(TypedDict, total=False):
    command: str
    note: str

llm = ChatOllama(model="qwen2.5:32b", temperature=0)

system_message = SystemMessage(
    content=(
        "你是一个便签编辑器。根据当前便签和修改要求，输出修改后的便签。"
        "只输出便签正文，不要解释，控制在 30 个汉字以内。"
    )
)

def edit_note(state: NoteState) -> NoteState:
    command = state.get("command")
    if command is None:
        raise ValueError("必须提供 command")

    old_note = state.get("note", "")
    user_message = HumanMessage(
        content=f"当前便签：{old_note or '空'}\n修改要求：{command}"
    )
    response = llm.invoke([system_message, user_message])
    new_note = response.content if isinstance(response.content, str) else str(response.content)
    return {"note": new_note}


builder = StateGraph(NoteState)
builder.add_node("edit_note", edit_note)
builder.add_edge(START, "edit_note")
builder.add_edge("edit_note", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

## 二、运行两次，得到两个版本

同一个 `thread_id` 表示同一条便签。

第二次调用时，不需要手动传入第一版 `note`。checkpoint 会把这个 `thread_id` 的旧 State 接回来。


In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "note-001"}}

first_input: NoteState = {"command": "写一条便签：周六上午去图书馆。"}
first_result = graph.invoke(first_input, config=config)
print("第一次：", first_result.get("note", ""))

second_input: NoteState = {"command": "补充：记得带借书卡。"}
second_result = graph.invoke(second_input, config=config)
print("第二次：", second_result.get("note", ""))

可以看到：

第一次输出的是第一版便签。此时 `thread_id = note-001` 下面还没有历史 State，节点只根据第一条 `command` 生成 `note`。

第二次仍然使用同一个 `thread_id`。LangGraph 会先从 checkpoint 里恢复上一轮的 `note`，再把新的 `command` 交给节点处理，所以输出变成了“上一版便签 + 新补充”。

重点：同一个 `thread_id` 让第二次运行拿到了第一次运行后的 State。


## 三、查看当前 checkpoint：`get_state`

`get_state(config)` 读取这个 `thread_id` 当前最新的 checkpoint。


In [ ]:
current_snapshot = graph.get_state(config)

print("当前 State：", current_snapshot.values)
print("下一步节点：", current_snapshot.next)

`current_snapshot.values` 就是当前最新 checkpoint 里的 State。

这里通常会看到两个字段：

- `command`：最近一次输入的修改要求；
- `note`：当前最新便签内容。

`current_snapshot.next == ()` 表示这个 checkpoint 已经运行完成，后面没有等待执行的节点。


## 四、查看历史 checkpoint：`get_state_history`

`get_state_history(config)` 会返回这个 `thread_id` 的 checkpoint 历史。

一次 `invoke` 内部会留下多个步骤级 checkpoint。为了只看每次运行完成后的版本，这里筛选 `next == ()` 的 checkpoint。


In [ ]:
history = list(graph.get_state_history(config))
completed_snapshots = [
    snapshot
    for snapshot in history
    if snapshot.next == () and snapshot.values.get("note")
]

for index, snapshot in enumerate(completed_snapshots, start=1):
    checkpoint_config = snapshot.config.get("configurable", {})
    checkpoint_id = checkpoint_config.get("checkpoint_id")
    print(index, checkpoint_id, snapshot.values.get("note", ""))

重点看历史 checkpoint 的顺序

get_state_history(config)返回的是这个 `thread_id` 的历史 checkpoint，默认顺序是从新到旧。

因此打印出来的编号含义是：

- 编号 1：最近的 checkpoint，也就是第二次运行完成后的版本；
- 编号 2：更早的 checkpoint，也就是第一次运行完成后的版本。

代码里筛选了 `snapshot.next == ()`，是为了只看“运行完成后的版本”，而不是一次 `invoke` 内部产生的中间步骤 checkpoint。


## 五、回到旧 checkpoint 继续运行

`thread_id` 找到一整段历史，`checkpoint_id` 找到历史里的某一个点。

下面取第一版便签对应的 checkpoint，从那里继续运行。


In [ ]:
first_version_snapshot = completed_snapshots[-1]
first_version_config: RunnableConfig = first_version_snapshot.config

branch_input: NoteState = {"command": "从第一版继续：改成下午去。"}
branch_result = graph.invoke(branch_input, config=first_version_config)

print(branch_result.get("note", ""))

`completed_snapshots[-1]` 取到的是最早那个完成版本，也就是第一版便签。

把这个 snapshot 的 `config` 传给 `graph.invoke(...)`，LangGraph 就会从那个历史 checkpoint 恢复 State，而不是从最新 State 开始。

所以这里的输出是在第一版便签基础上改出来的，不会带上第二版里“记得带借书卡”的内容。


## 六、人工修改 checkpoint：`update_state`

`update_state(config, values)` 可以手动改 State，并把修改结果保存成新的 checkpoint。

最简单的理解方式是：先看当前便签，再手动覆盖它，最后重新读取当前状态。


In [ ]:
before_update = graph.get_state(config)
print("修改前：", before_update.values.get("note", ""))

manual_update: NoteState = {"note": "人工改成最终版。"}
graph.update_state(config, manual_update)

after_update = graph.get_state(config)
print("修改后：", after_update.values.get("note", ""))

`update_state(config, manual_update)` 不会调用节点，也不会调用 LLM。它是直接把一份 State 更新写进 checkpoint。

这段输出分成两步看：

- `修改前`：读取当前最新 checkpoint 里的 `note`；
- `修改后`：手动覆盖 `note` 后，再读取最新 checkpoint。

这类操作适合人工审核、纠错、恢复状态等场景。

所以，checkpoint 的价值不是让代码变复杂，而是让你能检查、回退和修正图的状态。
